<a href="https://colab.research.google.com/github/rafaelrubo/extensao-cs-qualidade-do-ar/blob/main/extens%C3%A3o_qualidade_do_ar_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================
# INSTALAÇÃO DAS BIBLIOTECAS
# =========================================
!pip install pandas folium openpyxl -q

In [ ]:
# =========================================
# IMPORTAÇÃO DAS BIBLIOTECAS
# =========================================
import pandas as pd
import folium
import re
from google.colab import files

In [ ]:
# =========================================
# FUNÇÃO PARA CONVERTER DMS -> DECIMAL
# Exemplo:
# 23°38'43.1"S
# =========================================
def dms_to_decimal(dms):
    pattern = r"(\d+)°(\d+)'([\d\.]+)\"([NSEW])"
    match = re.match(pattern, dms)

    if not match:
        return None

    degrees, minutes, seconds, direction = match.groups()

    decimal = (
        float(degrees)
        + float(minutes) / 60
        + float(seconds) / 3600
    )

    if direction in ['S', 'W']:
        decimal *= -1

    return decimal

In [ ]:
# =========================================
# UPLOAD DO ARQUIVO
# =========================================
uploaded = files.upload()

arquivo = list(uploaded.keys())[0]

Saving 20270427 - medidas.xlsx to 20270427 - medidas (1).xlsx


In [ ]:
# =========================================
# LEITURA DA PLANILHA
# =========================================
df = pd.read_excel(
    arquivo,
    header=1
)

In [ ]:
# =========================================
# REMOVER LINHAS VAZIAS
# =========================================
df = df[df['Ponto'].notna()].copy()

In [ ]:
# =========================================
# CONVERTER COORDENADAS
# =========================================
df['lat'] = df['Latitude'].apply(dms_to_decimal)
df['lon'] = df['Longitude'].apply(dms_to_decimal)

In [ ]:
# =========================================
# CRIAR MAPA CENTRALIZADO
# =========================================
mapa = folium.Map(
    location=[df['lat'].mean(), df['lon'].mean()],
    zoom_start=10,
    tiles='OpenStreetMap'
)

In [ ]:
# =========================================
# ADICIONAR MARCADORES
# =========================================
for _, row in df.iterrows():

    popup = f"""
    <b>Ponto:</b> {row['Ponto']}<br>
    <b>Nome:</b> {row['Nome']}<br>
    <b>Ambiente:</b> {row['Ambiente']}<br>
    <b>CO₂:</b> {row['CO₂ (ppm)']} ppm<br>
    <b>HCHO:</b> {row['HCHO (mg/m³)']} mg/m³<br>
    <b>TVOC:</b> {row['TVOC (mg/m³)']} mg/m³<br>
    <b>Qualidade:</b> {row['Qualidade']}
    """

    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=popup,
        tooltip=row['Nome']
    ).add_to(mapa)

In [ ]:
# =========================================
# SALVAR MAPA
# =========================================
saida = 'mapa_localizacao.html'
mapa.save(saida)

print(f'Mapa salvo: {saida}')

Mapa salvo: mapa_localizacao.html


In [ ]:
# =========================================
# DOWNLOAD
# =========================================
files.download(saida)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>